# 🗣️ Entrenar a Filo (Conexiones)

Este cuaderno enseña a una IA chica (Qwen 0.5B) a entender lo que le dices a Filo durante la presentación y a contestar con órdenes de la lista (pasar fotos, música, video, volumen, zoom). **Nada de cuentas, PIN ni contraseñas.**

1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4** (gratis).
2. **Entorno de ejecución → Ejecutar todo.** Los pasos 1 a 5 entrenan solos (unos 20–30 min).
3. En el paso 6 pega tu **token de Hugging Face** tipo *Write* (https://huggingface.co/settings/tokens). Queda solo en Colab; no lo pegues en el chat.

In [ ]:
#@title 1. Preparar (instala lo necesario y baja las frases de entrenamiento)
!pip -q install peft==0.13.2 gguf sentencepiece
!rm -rf presentacion-56 && git clone -q --depth 1 https://github.com/quintillizasyuesugui-svg/presentacion-56
import json, torch, time, random
leer = lambda r: [json.loads(l) for l in open(r, encoding='utf-8') if l.strip()]
entrenar = leer('presentacion-56/entrenamiento-filo/datos/entrenar.jsonl')
prueba = leer('presentacion-56/entrenamiento-filo/datos/prueba.jsonl')
print(len(entrenar), 'frases para aprender ·', len(prueba), 'frases nuevas para el examen')
assert torch.cuda.is_available(), 'Falta la GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU'

In [ ]:
#@title 2. Cargar la IA base
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
BASE = 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
tok.padding_side = 'left'
modelo = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).cuda()
modelo = get_peft_model(modelo, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
modelo.print_trainable_parameters()
# Mismo formato que usa el Space: corto, para contestar rápido.
pedido = lambda f: f'Orden: {f}\nFilo:'

In [ ]:
#@title 3. Entrenar por épocas (cada época estudia todas las frases y después da un examen con frases nuevas)
EPOCAS = 3  #@param {type:"integer"}
LOTE = 32
FIN = tok.eos_token

def armar_lote(items):
    ids, etiquetas = [], []
    for x in items:
        p = tok(pedido(x['frase']))['input_ids']
        r = tok(' ' + x['salida'] + FIN)['input_ids']
        ids.append(p + r); etiquetas.append([-100] * len(p) + r)
    largo = max(map(len, ids))
    a = torch.full((len(ids), largo), tok.pad_token_id); e = torch.full((len(ids), largo), -100); m = torch.zeros((len(ids), largo), dtype=torch.long)
    for i, (x, y) in enumerate(zip(ids, etiquetas)):
        a[i, :len(x)] = torch.tensor(x); e[i, :len(y)] = torch.tensor(y); m[i, :len(x)] = 1
    return a.cuda(), e.cuda(), m.cuda()

@torch.no_grad()
def contestar(frases):
    modelo.eval(); salidas = []
    for i in range(0, len(frases), 64):
        enc = tok([pedido(f) for f in frases[i:i+64]], return_tensors='pt', padding=True).to('cuda')
        out = modelo.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tok.pad_token_id)
        salidas += [t.split('\n')[0].strip() for t in tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
    modelo.train(); return salidas

def examen():
    resp = contestar([x['frase'] for x in prueba])
    bien = sum(r == x['salida'] for r, x in zip(resp, prueba))
    dificiles = [i for i, x in enumerate(prueba) if x['clase'] == 'ia']
    bien_d = sum(resp[i] == prueba[i]['salida'] for i in dificiles)
    charla = [i for i, x in enumerate(prueba) if x['clase'] == 'nada']
    bien_c = sum(resp[i] == 'nada' for i in charla)
    return bien / len(prueba), bien_d / max(1, len(dificiles)), bien_c / max(1, len(charla)), resp

opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=2e-4)
pasos_total = EPOCAS * (len(entrenar) // LOTE)
plan = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.05, 1 - s / pasos_total))
mejor, mejor_pesos, notas = -1, None, []
modelo.train()
for epoca in range(1, EPOCAS + 1):
    random.seed(epoca); random.shuffle(entrenar); t0 = time.time(); perdida = 0
    for i in range(0, len(entrenar) - LOTE + 1, LOTE):
        a, e, m = armar_lote(entrenar[i:i+LOTE])
        with torch.autocast('cuda', dtype=torch.float16):
            loss = modelo(input_ids=a, attention_mask=m, labels=e).loss
        loss.backward(); opt.step(); plan.step(); opt.zero_grad(); perdida += loss.item()
    nota, nota_d, nota_c, resp = examen()
    notas.append({'epoca': epoca, 'bien': nota, 'dificiles': nota_d, 'charla': nota_c})
    print(f'Época {epoca}: {time.time()-t0:.0f} s · pérdida {perdida/(len(entrenar)//LOTE):.4f} · examen {nota*100:.1f} % bien · '
          f'difíciles (dos órdenes o nombres) {nota_d*100:.1f} % · charla bien ignorada {nota_c*100:.1f} %')
    if nota > mejor:
        mejor = nota; mejores_resp = resp
        mejor_pesos = {k: v.detach().clone() for k, v in modelo.state_dict().items() if 'lora' in k}
modelo.load_state_dict(mejor_pesos, strict=False)
print(f'Me quedo con la mejor época: {mejor*100:.1f} % bien en frases nuevas')
malas = [(x, r) for x, r in zip(prueba, mejores_resp) if r != x['salida']]
for x, r in malas[:20]:
    print('❌', x['frase'], '→', r, '| quería', x['salida'])

In [ ]:
#@title 4. Probar unas frases
for f in ['baja la musica y pasa a la siguiente', 'ponme algo tranquilo', 'pausa y despues dale play', 'que hora es en japon', 'hoy vamos a ver la fotosintesis']:
    t0 = time.time(); r = contestar([f])[0]; print(f'{f} → {r}  ({(time.time()-t0)*1000:.0f} ms en GPU)')

In [ ]:
#@title 5. Comprimir (archivo filo.gguf, unos 500 MB)
fusion = modelo.merge_and_unload(); fusion.save_pretrained('filo-hf', safe_serialization=True); tok.save_pretrained('filo-hf')
!rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!python llama.cpp/convert_hf_to_gguf.py filo-hf --outtype q8_0 --outfile filo.gguf
!ls -la filo.gguf
json.dump({'examen': mejor, 'epocas': notas}, open('nota.json', 'w'))
print('Listo para subir. Ahora el paso 6.')

In [ ]:
#@title 6. Entrar a Hugging Face (pega tu token tipo Write en el cuadro)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#@title 7. Subir a tu Hugging Face (modelo + Space gratis)
from huggingface_hub import HfApi
api = HfApi(); usuario = api.whoami()['name']
NOMBRE = 'filo-piensa'
repo = f'{usuario}/{NOMBRE}'
api.create_repo(repo, repo_type='model', exist_ok=True)
api.upload_file(path_or_fileobj='filo.gguf', path_in_repo='filo.gguf', repo_id=repo)
api.upload_file(path_or_fileobj='nota.json', path_in_repo='nota.json', repo_id=repo)
api.create_repo(repo, repo_type='space', space_sdk='docker', exist_ok=True)
api.add_space_variable(repo, 'MODELO', repo)
api.upload_folder(folder_path='presentacion-56/entrenamiento-filo/space', repo_id=repo, repo_type='space')
direccion = f"https://{usuario.lower().replace('_', '-')}-{NOMBRE}.hf.space"
print('Modelo:', f'https://huggingface.co/{repo}')
print('Space (tarda unos 5-10 min en armarse):', f'https://huggingface.co/spaces/{repo}')
print('Dirección para Conexiones:', direccion)

In [ ]:
#@title 8. Probar el Space (corre esto cuando el Space diga «Running»)
import requests
for f in ['baja la musica y pasa a la siguiente', 'pausa y despues dale play', 'que hora es en japon']:
    t0 = time.time(); r = requests.post(direccion + '/pensar', json={'frase': f}, timeout=90).json()
    print(f, '→', r['respuesta'], f"(IA {r['ms']} ms · total {(time.time()-t0)*1000:.0f} ms)")